In [3]:
# データを読み込んで形を確認
import pandas as pd

print(pd.read_excel(r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex1\89SEDEx1.xls", sheet_name=None))

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
{'Sheet1':     Time[sec.]     Interval   Interval.1   Interval.2      Interval.3  \
0   Time[sec.]  Nｺ Episodes  Nｺ Episodes  Nｺ Episodes  Time in [sec.]   
1   Time[sec.]       Freez.          Low         High          Freez.   
2           30            3            3            0            24.8   
3           60            4            4            0           29.22   
4           90            0            0            0              30   
5          120            1            1            0           29.98   
6          150            0            0            0              30   
7          180            2            2            0           28.88   
8          210            0            0            0              30   
9          240            0            0            0              30   
10         270            2            2            0           29.92   
11         300            1            1

In [ ]:
# RのコードをPythonに変換 
import pandas as pd


def read_per3_bin(file_path: str, name: str) -> pd.DataFrame: # 3分毎のbinごとにデータを読み込む
    df = pd.read_excel(file_path) # データを読み込む

    bins = [ # 3分毎のbinの範囲とラベル
        (3, 8, "3"),
        (9, 14, "6"),
        (15, 20, "9"),
        (21, 26, "12"),
        (27, 32, "15"),
    ]

    frames = [] # 結果を格納するリスト
    for start, end, time_label in bins: # 3分毎のbinごとにデータを処理
        tmp = (
            df.iloc[:, 4]           # 5列目（0-indexed）
            .iloc[start - 1 : end]  # 行 3:8, 9:14, ...
            .astype(float)          # 数値に変換
            .mul(100 / 30 )         # 30秒ごとのデータを3分ごとの平均に変換
        )

        frames.append( # 結果をリストに追加
            pd.DataFrame({  # データフレームを作成
                "Freezing": [tmp.mean()], # 平均値を計算
                "Time": [time_label], # ラベルを追加
                "No": [name], # 名前を追加
            })
        )

    return pd.concat(frames, ignore_index=True)

# 例:
tmp = read_per3_bin("path/to/file.xls", "animal_name")

In [ ]:
# 3分毎の折れ線グラフ用のFear Extinctionデータの読み込み

from pathlib import Path  # noqa: F811

import numpy as np  # noqa: F811
import pandas as pd

# 自動でエクセルファイルを読み込む

folder_Ex1 = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex1"  # フォルダのパスを指定
folder_Ex2 = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex2"  # フォルダのパスを指定

files_Ex1 = sorted(Path(folder_Ex1).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート
files_Ex2 = sorted(Path(folder_Ex2).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート

if not files_Ex1 and not files_Ex2:
    print("指定されたフォルダに .xls ファイルはありません")
else:
    dfs = [] # 空のリストを作成して、各ファイルのデータフレームを格納
    for file in files_Ex1 + files_Ex2: # ファイルごとにループ
        df = (
        pd.read_excel(file) 
        .iloc[2:8, [4]]  # 2行目から7行目まで、4列目を抽出
        .assign(Freezing = lambda df: df['Interval.3'] / 60 * 100 ) # Freezing Time (%) を計算し列に追加
        [['Freezing']] # Freezing列のみを残す
        .assign(
        Time  = lambda df: list(range(1, len(df) + 1)), # Time列に1から行数までの連番を追加
        No = lambda df: Path(file).stem,  # No列にpathからファイル名を抽出して追加
        Group = lambda df: np.select( # Group列に条件に応じた値を追加
            condlist=[ # 条件のリスト：No列に各文字列が含まれているか
                df['No'].str.contains('SED'),
                df['No'].str.contains('LIE'),
                df['No'].str.contains('MOE')
            ],
            choicelist=['SED', 'LIE', 'MOE'], # 条件にマッチしたときに入れる値のリスト
            default='Other' # どれにも当てはまらない場合のデフォルト値
            )
        )
     )
        dfs.append(df) # データフレームをリストに追加

    dataFC = pd.concat(dfs, ignore_index=True) # リスト内のデータフレームを縦に結合して1つのデータフレームにする
    print(f"{len(files_Ex1 + files_Ex2)} 件の .xls ファイルを読み込みました") # 読み込んだファイル数を表示
    print(dataFC) # データフレームの内容を表示

In [ ]:
# 上の関数と自動読み込みのコードをハイブリットしたコード

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# 旧形式 .xls の OLE2 警告を抑制
# warnings.filterwarnings("ignore", message=".*OLE2 inconsistency.*")

def infer_group(name: str) -> str: # ファイル名からグループを推測する関数
    if "SED" in name:
        return "SED"
    elif "LIE" in name:
        return "LIE"
    elif "MOE" in name:
        return "MOE"
    return "Other"

def read_extinction_per3(files):   # 3分ごとのデータを読み込む関数
    rows = [] # 空のリストを作成して、各ファイルのデータを格納

    for file in files:
        # 旧 .xls では pandas で警告が出ることがあるので抑制
        # with warnings.catch_warnings():
            # warnings.filterwarnings("ignore", message=".*OLE2 inconsistency.*")
            # df = pd.read_excel(file)

        col5 = pd.to_numeric(df.iloc[:, 4], errors="coerce")  # 5列目（E列）
        bins = [ # 3分毎のbinの範囲とラベル
            (3, 8, "3"),
            (9, 14, "6"),
            (15, 20, "9"),
            (21, 26, "12"),
            (27, 32, "15"),
        ]

        for start, end, time_label in bins: # 3分毎のbinごとにデータを処理
            freezing = (
                col5.iloc[start - 1:end] # 3:8, 9:14, ...
                .astype(float)           # 数値に変換
                .mul(100 / 30)           # 30秒ごとのデータを3分ごとの平均に変換
                .mean()                  # 平均値を計算  
            )
            rows.append({
                "No": Path(file).stem,   # ファイル名を追加
                "Time": time_label,      # ラベルを追加
                "Freezing": freezing,    # 平均値を追加
                "Group": infer_group(Path(file).stem),  # グループを推測して追加
            })

    return pd.DataFrame(rows)

# 既存のフォルダ
folder_Ex1 = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex1"
folder_Ex2 = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex2"

files_Ex1 = sorted(Path(folder_Ex1).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート
files_Ex2 = sorted(Path(folder_Ex2).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート

# 3分ごとのデータ
if not files_Ex1 and not files_Ex2:
    print("指定されたフォルダに .xls ファイルはありません")
else:
    dataEx1_per3 = read_extinction_per3(files_Ex1) # Ex1の3分ごとのデータを読み込む
    dataEx2_per3 = read_extinction_per3(files_Ex2) # Ex2の3分ごとのデータを読み込む

    print(f"Ex1: {len(files_Ex1)} 件, Ex2: {len(files_Ex2)} 件") 
    print(dataEx1_per3.head()) 
    print(dataEx2_per3.head())

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but 